# Couche 3 — XGBoost supervisé (IA-6)
**CyberGuardian AI** · Notebook expérimentation · Août 2026

---

## Objectif

Ce notebook permet d'**explorer, tester, évaluer et ajuster** la Couche 3 du moteur de scoring.  
Il ne fait **pas partie de l'API** — c'est un outil d'expérimentation.

## Ce qu'on peut faire ici

1. **Charger et inspecter le dataset supervisé** — toutes classes, distribution fraudes/légitimes
2. **Entraîner XGBoost** — ajuster `max_depth`, `learning_rate`, `scale_pos_weight`
3. **Validation croisée** — voir la stabilité sur 5 folds stratifiés
4. **Évaluer les performances** — AUC-PR, rappel @1% FPR, matrices de confusion
5. **Explicabilité SHAP** — quelles features expliquent les décisions ?
6. **Champion/challenger** — comparer deux entraînements
7. **Tester predict() en direct** — scorer des transactions réelles depuis Redis

## Prérequis

```bash
# Docker doit tourner avec Redis et MinIO
docker compose up redpanda redis postgres minio redpanda-init minio-init -d
# Le simulateur doit avoir tourné
docker compose --profile simulator run --rm simulator --mode batch --reset-profiles
```

---
## 0. Imports et configuration

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

os.environ['REDIS_HOST']       = 'localhost'
os.environ['REDIS_PORT']       = '16379'
os.environ['MINIO_ENDPOINT']   = 'localhost:19000'
os.environ['MINIO_ACCESS_KEY'] = 'minioadmin'
os.environ['MINIO_SECRET_KEY'] = 'minioadmin123'

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = 'plotly_white'

from simulator.subscribers  import generate_subscribers
from simulator.calendrier   import planifier_simulation
from simulator.config       import SEED, NB_ABONNES
from engine.supervised.dataset import build_dataset, XGB_FEATURE_NAMES

print('Bibliothèques chargées')

Bibliothèques chargées


---
## 1. Charger les données

In [2]:
print(f'Génération de {NB_ABONNES} comptes (seed={SEED})...')
comptes    = generate_subscribers(NB_ABONNES, seed=SEED)
scenarios  = planifier_simulation(comptes, seed=SEED)
evenements = [ev for sc in scenarios for ev in sc.evenements]
events     = [e.payload for e in evenements if e.stream == 'transactions']

n_fraude  = sum(1 for e in events if e.get('label_fraude') == 1)
n_legitime = len(events) - n_fraude
ratio      = n_legitime / max(n_fraude, 1)
print(f'{len(events)} transactions')
print(f'  Fraudes   : {n_fraude} ({100*n_fraude/len(events):.1f}%)')
print(f'  Légitimes : {n_legitime} ({100*n_legitime/len(events):.1f}%)')
print(f'  Ratio légitimes/fraudes = {ratio:.1f}  → scale_pos_weight ≈ {ratio:.0f}')

Génération de 500 comptes (seed=42)...
Simulation 30 jours planifiée :
  Scénarios :  22793  (fraudes=220, légitimes=22573)
  Événements:  23934
23314 transactions
  Fraudes   : 749 (3.2%)
  Légitimes : 22565 (96.8%)
  Ratio légitimes/fraudes = 30.1  → scale_pos_weight ≈ 30


In [3]:
ds = build_dataset(events=events)

print(f'Dataset supervisé :')
print(f'  Train : {ds.n_train} (fraudes={int(ds.y_train.sum())}, légitimes={ds.n_train-int(ds.y_train.sum())})')
print(f'  Test  : {ds.n_test}  (fraudes={int(ds.y_test.sum())})')
print(f'  scale_pos_weight = {ds.scale_pos_weight:.1f}')
print(f'  Features ({len(ds.feature_names)}) : {ds.feature_names}')

Dataset supervisé :
  Train : 18512 (fraudes=575, légitimes=17937)
  Test  : 4802  (fraudes=174)
  scale_pos_weight = 31.2
  Features (17) : ['amount_ratio', 'zscore_montant', 'hours_since_sim_swap', 'new_device', 'new_beneficiary', 'is_roaming', 'otp_count_1h', 'nb_tx_1h', 'nb_tx_24h', 'nb_tx_7j', 'nb_otp_24h', 'nb_swaps_30j', 'is_active_hour', 'hour_of_day', 'montant_moyen', 'ecart_type_montant', 'solde']


---
## 2. Explorer la distribution des features — fraudes vs légitimes

In [4]:
# Combiner train + test pour la visualisation
X_all = np.vstack([ds.X_train, ds.X_test])
y_all = np.concatenate([ds.y_train, ds.y_test])

df_all = pd.DataFrame(X_all, columns=ds.feature_names)
df_all['label'] = ['Fraude' if y == 1 else 'Légitime' for y in y_all]

# Ratio fraude/légitime par feature
df_mean = df_all.groupby('label')[ds.feature_names].mean().T.round(4)
df_mean['ratio'] = (df_mean['Fraude'] / df_mean['Légitime'].replace(0, 0.001)).round(2)
df_mean = df_mean.sort_values('ratio', ascending=False)
print('Features les plus discriminantes (ratio moyen fraude/légitime) :')
df_mean

Features les plus discriminantes (ratio moyen fraude/légitime) :


label,Fraude,Légitime,ratio
new_beneficiary,1.000000,0.000000,1000.00
new_device,1.000000,0.001100,909.09
ecart_type_montant,47794.312500,28605.654297,1.67
montant_moyen,109639.703125,68976.664062,1.59
is_roaming,1.000000,0.708000,1.41
solde,302286.531250,219994.515625,1.37
amount_ratio,0.230900,0.195200,1.18
hours_since_sim_swap,9999.000000,9999.000000,1.00
hour_of_day,12.907900,12.974400,0.99
is_active_hour,0.974600,1.000000,0.97


In [5]:
# Boxplots comparatifs pour les features clés
top_features = df_mean.head(6).index.tolist()
fig = make_subplots(rows=2, cols=3, subplot_titles=top_features)
for i, feat in enumerate(top_features):
    row, col = divmod(i, 3)
    for label, color in [('Fraude', 'crimson'), ('Légitime', 'steelblue')]:
        vals = df_all[df_all['label'] == label][feat]
        fig.add_trace(
            go.Box(y=vals, name=label, marker_color=color, showlegend=(i==0)),
            row=row+1, col=col+1
        )
fig.update_layout(height=600, title_text='Distribution features discriminantes — Fraude vs Légitime')
fig.show()

In [6]:
# Distribution de amount_ratio
fig = px.histogram(
    df_all[df_all['amount_ratio'] < 20],
    x='amount_ratio', color='label', nbins=50, barmode='overlay', opacity=0.7,
    color_discrete_map={'Fraude': 'crimson', 'Légitime': 'steelblue'},
    title='Distribution du ratio montant (montant / montant_habituel)'
)
fig.add_vline(x=3, line_dash='dash', annotation_text='Seuil règle R01 (×3)')
fig.add_vline(x=5, line_dash='dot',  annotation_text='Seuil règle R06 (×5)')
fig.show()

---
## 3. Entraîner XGBoost

**Paramètres clés à ajuster :**
- `max_depth` : profondeur des arbres. Plus profond = plus puissant mais risque d'overfitting
- `learning_rate` : pas d'apprentissage. Plus petit = plus stable, nécessite plus d'arbres
- `scale_pos_weight` : poids des fraudes. `auto` = calculé depuis le dataset

**Astuce** : commence par les valeurs par défaut, puis ajuste après avoir vu les métriques.

In [7]:
# ── Paramètres ajustables ────────────────────────────────
N_ESTIMATORS       = 300    # ← 100, 200, 300, 500
MAX_DEPTH          = 6      # ← 4, 6, 8
LEARNING_RATE      = 0.05   # ← 0.01, 0.05, 0.1
SCALE_POS_WEIGHT   = 'auto' # ← 'auto' ou valeur numérique ex: 20, 50
# ────────────────────────────────────────────────────────

os.environ['XGB_N_ESTIMATORS']  = str(N_ESTIMATORS)
os.environ['XGB_MAX_DEPTH']     = str(MAX_DEPTH)
os.environ['XGB_LEARNING_RATE'] = str(LEARNING_RATE)

from interfaces.store import ObjectStore
import pickle

class MemStore(ObjectStore):
    def __init__(self): self._d = {}
    def upload(self, b, k, data): self._d[f'{b}/{k}'] = data
    def download(self, b, k):
        key = f'{b}/{k}'
        if key not in self._d: raise KeyError(key)
        return self._d[key]
    def exists(self, b, k): return f'{b}/{k}' in self._d

store_exp = MemStore()

from engine.supervised.train import train
res = train(ds, store=store_exp, promote=True)

print('═' * 50)
print(f'AUC-PR test    : {res.metrics["auc_pr_test"]:.4f}')
print(f'CV AUC-PR mean : {res.metrics["cv_auc_pr_mean"]:.4f} ± {res.metrics["cv_auc_pr_std"]:.4f}')
print(f'scale_pos_weight utilisé : {res.metrics["scale_pos_weight"]:.1f}')
print('═' * 50)

══════════════════════════════════════════════════
AUC-PR test    : 1.0000
CV AUC-PR mean : 1.0000 ± 0.0000
scale_pos_weight utilisé : 31.2
══════════════════════════════════════════════════


---
## 4. Évaluation complète

In [8]:
from engine.supervised.detector import XGBoostDetector
from engine.supervised.evaluate import evaluate, _batch_score

detector = XGBoostDetector(store=store_exp)
report   = evaluate(detector, ds, store=store_exp, save_report=False)

print('─' * 50)
print(f'AUC-PR                  : {report["auc_pr"]:.4f}')
print(f'Rappel @ 1% FPR         : {report["recall_at_1pct_fpr"]:.4f}')
print(f'Rappel @ 5% FPR         : {report["recall_at_5pct_fpr"]:.4f}')
print(f'Précision @ rappel 50%  : {report["precision_at_recall_50"]:.4f}')
print(f'Précision @ rappel 80%  : {report["precision_at_recall_80"]:.4f}')
print(f'Score moyen fraudes     : {report["score_mean_fraud"]:.1f} / 100')
print(f'Score moyen légitimes   : {report["score_mean_legit"]:.1f} / 100')
print(f'Séparation              : {report["score_separation"]:.1f} points')
print('─' * 50)

──────────────────────────────────────────────────
AUC-PR                  : 1.0000
Rappel @ 1% FPR         : 1.0000
Rappel @ 5% FPR         : 1.0000
Précision @ rappel 50%  : 1.0000
Précision @ rappel 80%  : 1.0000
Score moyen fraudes     : 100.0 / 100
Score moyen légitimes   : 0.0 / 100
Séparation              : 100.0 points
──────────────────────────────────────────────────


In [9]:
# Distribution des probabilités — fraudes vs légitimes
probas = _batch_score(detector, ds)
scores = [p * 100 for p in probas]

df_scores = pd.DataFrame({
    'score': scores,
    'label': ['Fraude' if y == 1 else 'Légitime' for y in ds.y_test]
})

fig = px.histogram(
    df_scores, x='score', color='label', nbins=50,
    barmode='overlay', opacity=0.7,
    color_discrete_map={'Fraude': 'crimson', 'Légitime': 'steelblue'},
    title='Distribution des scores XGBoost — Fraudes vs Légitimes',
    labels={'score': 'Score (0-100)', 'count': 'Nb transactions'}
)
for seuil, color in [(30, 'orange'), (70, 'red'), (90, 'darkred')]:
    fig.add_vline(x=seuil, line_dash='dash', line_color=color,
                  annotation_text=f'Seuil {seuil}')
fig.show()

In [10]:
# Matrice de confusion aux 4 seuils
df_cm = pd.DataFrame([
    {'Seuil': int(s), **cm}
    for s, cm in report['confusion_by_threshold'].items()
])
df_cm

,Seuil,tp,fp,tn,fn,precision,recall,fpr
0,30,174,0,4628,0,1.0,1.0,0.0
1,50,174,0,4628,0,1.0,1.0,0.0
2,70,174,0,4628,0,1.0,1.0,0.0
3,90,174,0,4628,0,1.0,1.0,0.0


---
## 5. Explicabilité SHAP

In [11]:
# Importance SHAP globale (déjà calculée par evaluate)
if report.get('shap_importance'):
    df_shap = pd.DataFrame(report['shap_importance'])
    fig = px.bar(
        df_shap, x='importance', y='feature', orientation='h',
        title='Importance SHAP globale — Top features',
        color='importance', color_continuous_scale='Reds'
    )
    fig.update_layout(yaxis={'categoryorder': 'total ascending'})
    fig.show()
else:
    print('SHAP non disponible')

In [12]:
# SHAP sur un exemple individuel — fraude
import random
from datetime import datetime, timezone
from simulator.scenarios import build_sim_swap_simple

c_f  = generate_subscribers(1, seed=77)[0]
ts_f = datetime(2026, 7, 14, 9, 0, tzinfo=timezone.utc)
sc_f = build_sim_swap_simple(c_f, random.Random(77), ts_f)
ev_f = next(e.payload for e in sc_f.evenements if e.stream == 'transactions')
pf   = {
    'montant_moyen_habituel': c_f.montant_moyen_habituel,
    'montant_moyen': c_f.montant_moyen_habituel,
    'ecart_type_montant': c_f.ecart_type_montant,
    'devices_connus': [c_f.device_id_habituel], 'beneficiaires_connus': [],
    'antennes_connues': [c_f.antenne_domicile], 'antenne_domicile': c_f.antenne_domicile,
    'ts_dernier_swap': ts_f.isoformat(), 'nb_otp_1h': 6, 'nb_tx_1h': 3,
    'nb_tx_24h': 5, 'nb_tx_7j': 20, 'nb_otp_24h': 6, 'nb_swaps_30j': 1,
    'solde': c_f.solde, 'heures_actives': c_f.heures_actives,
    'fenetre_1h_ts': [], 'nb_transactions': 20,
}

r_f = detector.predict(ev_f, pf)
print(f'Score : {r_f.score}/100  proba={r_f.probability:.4f}')
print(f'SHAP top-3 :')
for s in r_f.shap_top3:
    print(f'  {s["feature"]:<30} val={s["value"]:.3f}  shap={s["shap"]:+.4f}  [{s["direction"]}]')

# Visualiser le SHAP de cet exemple
df_shap_ex = pd.DataFrame(r_f.shap_top3)
colors = ['crimson' if d == 'fraud' else 'steelblue' for d in df_shap_ex['direction']]
fig = go.Figure(go.Bar(
    x=df_shap_ex['shap'], y=df_shap_ex['feature'],
    orientation='h', marker_color=colors
))
fig.update_layout(
    title=f'SHAP — SIM_SWAP_SIMPLE (score={r_f.score})',
    xaxis_title='Contribution SHAP (+ = vers fraude)',
    height=300
)
fig.show()

Score : 100/100  proba=0.9999
SHAP top-3 :
  new_beneficiary                val=1.000  shap=+7.9599  [fraud]
  new_device                     val=1.000  shap=+1.4762  [fraud]
  is_roaming                     val=1.000  shap=+0.0414  [fraud]


---
## 6. Champion / Challenger — comparer deux configurations

In [13]:
from engine.supervised.train import train as train_xgb

# Configuration A (conservatrice)
os.environ['XGB_N_ESTIMATORS']  = '200'
os.environ['XGB_MAX_DEPTH']     = '4'
os.environ['XGB_LEARNING_RATE'] = '0.05'
store_A = MemStore()
res_A   = train_xgb(ds, store=store_A, promote=True)

# Configuration B (plus profonde)
os.environ['XGB_N_ESTIMATORS']  = '300'
os.environ['XGB_MAX_DEPTH']     = '6'
os.environ['XGB_LEARNING_RATE'] = '0.03'
store_B = MemStore()
res_B   = train_xgb(ds, store=store_B, promote=True)

print(f'{'Config':<12} {'AUC-PR':>10} {'CV mean':>10} {'CV std':>8}')
print('─' * 44)
print(f'A (d=4, lr=0.05) {res_A.metrics["auc_pr_test"]:>10.4f} {res_A.metrics["cv_auc_pr_mean"]:>10.4f} {res_A.metrics["cv_auc_pr_std"]:>8.4f}')
print(f'B (d=6, lr=0.03) {res_B.metrics["auc_pr_test"]:>10.4f} {res_B.metrics["cv_auc_pr_mean"]:>10.4f} {res_B.metrics["cv_auc_pr_std"]:>8.4f}')

champion = 'A' if res_A.metrics['auc_pr_test'] >= res_B.metrics['auc_pr_test'] else 'B'
print(f'\n→ Champion : Configuration {champion}')

Config           AUC-PR    CV mean   CV std
────────────────────────────────────────────
A (d=4, lr=0.05)     1.0000     1.0000   0.0000
B (d=6, lr=0.03)     1.0000     1.0000   0.0000

→ Champion : Configuration A


---
## 7. Tester predict() sur des profils Redis réels

In [15]:
import json
try:
    import redis as _redis
    r = _redis.Redis(host='localhost', port=16379, decode_responses=True)
    keys = r.keys('profile:*')
    print(f'{len(keys)} profils dans Redis')

    # Prendre 5 profils aléatoires et les scorer
    import random
    from datetime import datetime, timezone, timedelta

    # Utiliser le modèle sauvegardé dans MinIO si disponible
    det_real = XGBoostDetector()  # charge depuis MinIO
    if not det_real.is_ready:
        det_real = detector  # fallback sur le modèle en mémoire
        print('Fallback : modèle en mémoire (MinIO non disponible)')

    resultats = []
    for key in random.sample(keys, min(5, len(keys))):
        profil = json.loads(r.get(key))
        ts_swap = (datetime.now(timezone.utc) - timedelta(minutes=10)).isoformat()
        event = {
            'id_compte':       profil['id_compte'],
            'horodatage':      datetime.now(timezone.utc).isoformat(),
            'montant':         profil.get('montant_moyen_habituel', 10000) * 4,
            'device_id':       'DEV-TEST',
            'id_beneficiaire': 'BEN-TEST',
            'antenne':         'MAT-ANT-999',
        }
        profil_sim = dict(profil)
        profil_sim['ts_dernier_swap'] = ts_swap
        profil_sim['nb_otp_1h']       = 5
        res_r = det_real.predict(event, profil_sim)
        decision = 'BLOCK' if res_r.score >= 70 else ('CHALLENGE' if res_r.score >= 30 else 'PASS')
        resultats.append({
            'id_compte': profil['id_compte'][:20],
            'segment':   profil.get('segment'),
            'score':     res_r.score,
            'proba':     round(res_r.probability, 4),
            'décision':  decision,
            'shap_top1': res_r.shap_top3[0]['feature'] if res_r.shap_top3 else ''
        })

    pd.DataFrame(resultats)

except Exception as e:
    print(f'Redis non disponible : {e}')
    print('Lance Docker pour tester avec des profils réels.')

500 profils dans Redis


---
## 8. Sauvegarder le meilleur modèle dans MinIO

In [ ]:
# ── Décommenter pour sauvegarder dans MinIO ──────────────
# SAVE_TO_MINIO = True
# if SAVE_TO_MINIO:
#     # Réappliquer les paramètres du meilleur config
#     os.environ['XGB_N_ESTIMATORS']  = '300'
#     os.environ['XGB_MAX_DEPTH']     = '6'
#     os.environ['XGB_LEARNING_RATE'] = '0.03'
#     res_final = train_xgb(ds, promote=True)  # utilise MinIO réel
#     print(f'Sauvegardé → {res_final.model_key}')
#     print(f'Champion : {res_final.is_champion}')
#     print(f'AUC-PR   : {res_final.metrics["auc_pr_test"]:.4f}')
print('Cellule désactivée — décommenter pour sauvegarder dans MinIO')

---
## Tableau de bord expérimental

| Config | n_estimators | max_depth | lr | AUC-PR | Rappel@1%FPR | Décision |
|---|---|---|---|---|---|---|
| Défaut | 300 | 6 | 0.05 | ... | ... | ... |
| Config A | 200 | 4 | 0.05 | ... | ... | ... |
| Config B | 300 | 6 | 0.03 | ... | ... | ... |

> Remplir ce tableau au fur et à mesure des expériences.